# Gold province reporting-area dimensions

## Purpose

Publish the D-57 geography and its selected boundary for Analytics while keeping `dim_region` unchanged.

## Grain

`dim_province_reporting_area` has one governed reporting area per PSGC version plus Unknown. The boundary dimension has one selected boundary-status row per reporting area and boundary version.


## 1. Gate validated Silver inputs


In [0]:
%sql
USE CATALOG `buildabida-capstone`;
CREATE SCHEMA IF NOT EXISTS `03-gold`;

DECLARE OR REPLACE VARIABLE dimension_rule_version STRING
    DEFAULT 'gold-dim-province-reporting-area-v1';
DECLARE OR REPLACE VARIABLE boundary_rule_version STRING
    DEFAULT 'gold-dim-province-reporting-area-boundary-v1';
DECLARE OR REPLACE VARIABLE boundary_crs STRING DEFAULT 'EPSG:4326';

SELECT ASSERT_TRUE(
    COUNT(*) > 0 AND COUNT(DISTINCT run_id) = 1
    AND COUNT(*) = COUNT(DISTINCT reporting_area_key)
    AND COUNT_IF(reporting_area_key = 0) = 1,
    'STOP: Silver reporting-area reference is empty, mixed-run, or unsafe'
)
FROM `02-silver`.silver_province_reporting_area;

SELECT ASSERT_TRUE(
    COUNT(*) > 0
    AND COUNT_IF(action = 'stop' AND status IN ('FAIL', 'ERROR')) = 0,
    'STOP: Silver province-reporting validation is missing or unsafe'
)
FROM `04-validation`.silver_dq_results
WHERE table_name IN (
    'silver_province_reporting_area',
    'silver_province_reporting_area_boundary'
)
  AND validation_run_id = (
      SELECT MAX_BY(validation_run_id, run_ts)
      FROM `04-validation`.silver_dq_results
      WHERE table_name = 'silver_project_province_map'
        AND pipeline_run_id = (
            SELECT MAX(run_id) FROM `02-silver`.silver_project_province_map
        )
  );

CREATE OR REPLACE TEMPORARY VIEW gold_reporting_area_versions AS
WITH area AS (
    SELECT MAX(run_id) AS silver_area_run_id, MAX(psgc_run_id) AS psgc_run_id
    FROM `02-silver`.silver_province_reporting_area
),
boundary AS (
    SELECT MAX(run_id) AS silver_boundary_run_id,
        MAX(boundary_run_id) AS boundary_run_id,
        MAX(boundary_version) AS boundary_version
    FROM `02-silver`.silver_province_reporting_area_boundary
)
SELECT
    area.silver_area_run_id, area.psgc_run_id,
    boundary.silver_boundary_run_id, boundary.boundary_run_id,
    boundary.boundary_version,
    SHA2(CONCAT_WS(
        '||', area.silver_area_run_id, boundary.silver_boundary_run_id, dimension_rule_version
    ), 256) AS dimension_run_id,
    SHA2(CONCAT_WS(
        '||', area.silver_area_run_id, boundary.silver_boundary_run_id, boundary_rule_version
    ), 256) AS gold_boundary_run_id
FROM area
CROSS JOIN boundary;


## 2. Create dimensions and deterministic sources


In [0]:
%sql
CREATE TABLE IF NOT EXISTS `03-gold`.dim_province_reporting_area (
    reporting_area_key BIGINT NOT NULL COMMENT 'Deterministic D-33 key. Key 0 is Unknown or Unmapped.',
    psgc_reporting_area_code STRING COMMENT 'Official PSGC province, independent-city, or NCR code. NULL for key 0.',
    reporting_area_name STRING COMMENT 'Official reporting-area name or Unknown or unmapped.',
    reporting_area_short_name STRING COMMENT 'Display label for the reporting area.',
    reporting_area_type STRING NOT NULL COMMENT 'PROVINCE, INDEPENDENT_CITY, NCR, or UNKNOWN.',
    region_key BIGINT NOT NULL COMMENT 'Parent region key. Key 0 for Unknown.',
    psgc_version STRING COMMENT 'Selected PSGC publisher version.',
    is_geographic_area BOOLEAN NOT NULL COMMENT 'False only for the Unknown member.',
    centroid_latitude DOUBLE COMMENT 'Latitude of the valid selected boundary centroid. NULL when unavailable.',
    centroid_longitude DOUBLE COMMENT 'Longitude of the valid selected boundary centroid. NULL when unavailable.',
    run_id STRING NOT NULL COMMENT 'Deterministic Gold dimension run ID.',
    source_load_ts TIMESTAMP COMMENT 'Latest selected PSGC or boundary load time.',
    psgc_run_id STRING NOT NULL COMMENT 'Silver PSGC run behind the reporting area.',
    boundary_run_id STRING NOT NULL COMMENT 'Silver selected boundary ingest run.'
)
USING DELTA
COMMENT 'Gold D-57 province reporting-area dimension. One official reporting area per PSGC version plus Unknown.';

CREATE TABLE IF NOT EXISTS `03-gold`.dim_province_reporting_area_boundary (
    boundary_key BIGINT NOT NULL COMMENT 'Deterministic key for reporting area and boundary version.',
    reporting_area_key BIGINT NOT NULL COMMENT 'Joins to dim_province_reporting_area.',
    boundary_geometry GEOMETRY(4326) COMMENT 'Selected reporting-area geometry in EPSG:4326. NULL when unavailable or unsafe.',
    boundary_version STRING COMMENT 'Selected boundary publisher version.',
    coordinate_reference_system STRING NOT NULL COMMENT 'EPSG:4326.',
    geometry_valid_flag BOOLEAN NOT NULL COMMENT 'True only for a valid published geometry.',
    boundary_status STRING NOT NULL COMMENT 'VALID_BOUNDARY, MISSING_BOUNDARY, AMBIGUOUS_BOUNDARY, INVALID_BOUNDARY, or NOT_APPLICABLE.',
    run_id STRING NOT NULL COMMENT 'Deterministic Gold boundary run ID.',
    source_load_ts TIMESTAMP COMMENT 'Latest selected boundary load time.',
    psgc_run_id STRING NOT NULL COMMENT 'Silver PSGC run behind the reporting area.',
    boundary_run_id STRING NOT NULL COMMENT 'Silver selected boundary ingest run.'
)
USING DELTA
COMMENT 'One selected boundary-status row per D-57 reporting area and boundary version.';

CREATE OR REPLACE TEMPORARY VIEW dim_province_reporting_area_source AS
SELECT
    area.reporting_area_key, area.psgc_reporting_area_code, area.reporting_area_name,
    area.reporting_area_short_name, area.reporting_area_type,
    COALESCE(region.region_key, CAST(0 AS BIGINT)) AS region_key,
    area.psgc_version, area.is_geographic_area,
    CASE WHEN boundary.boundary_status = 'VALID_BOUNDARY'
        THEN ST_Y(ST_CENTROID(ST_SETSRID(
            ST_GEOMFROMWKB(boundary.boundary_geometry_wkb), boundary.boundary_srid
        ))) END AS centroid_latitude,
    CASE WHEN boundary.boundary_status = 'VALID_BOUNDARY'
        THEN ST_X(ST_CENTROID(ST_SETSRID(
            ST_GEOMFROMWKB(boundary.boundary_geometry_wkb), boundary.boundary_srid
        ))) END AS centroid_longitude,
    versions.dimension_run_id AS run_id,
    GREATEST(area.source_load_ts, boundary.source_load_ts) AS source_load_ts,
    area.psgc_run_id, boundary.boundary_run_id
FROM `02-silver`.silver_province_reporting_area AS area
INNER JOIN `02-silver`.silver_province_reporting_area_boundary AS boundary
    ON area.reporting_area_key = boundary.reporting_area_key
CROSS JOIN gold_reporting_area_versions AS versions
LEFT JOIN `03-gold`.dim_region AS region
    ON area.parent_region_code = region.psgc_region_code
   AND area.psgc_version <=> region.psgc_version;

SELECT ASSERT_TRUE(
    COUNT(*) = COUNT(DISTINCT reporting_area_key)
    AND COUNT_IF(reporting_area_key = 0) = 1
    AND COUNT_IF(region_key IS NULL) = 0
    AND COUNT_IF(reporting_area_type <> 'UNKNOWN' AND region_key = 0) = 0,
    'STOP: reporting-area dimension keys, Unknown member, or parent regions are unsafe'
)
FROM dim_province_reporting_area_source;

CREATE OR REPLACE TEMPORARY VIEW dim_province_reporting_area_boundary_source AS
SELECT
    boundary.boundary_key, boundary.reporting_area_key,
    CASE WHEN boundary.boundary_status = 'VALID_BOUNDARY'
        THEN ST_SETSRID(ST_GEOMFROMWKB(
            boundary.boundary_geometry_wkb
        ), boundary.boundary_srid) END AS boundary_geometry,
    boundary.boundary_version, boundary_crs AS coordinate_reference_system,
    boundary.boundary_status = 'VALID_BOUNDARY' AS geometry_valid_flag,
    boundary.boundary_status, versions.gold_boundary_run_id AS run_id,
    boundary.source_load_ts, boundary.psgc_run_id, boundary.boundary_run_id
FROM `02-silver`.silver_province_reporting_area_boundary AS boundary
CROSS JOIN gold_reporting_area_versions AS versions;


## 3. Merge idempotently


In [0]:
%sql
MERGE INTO `03-gold`.dim_province_reporting_area AS target
USING dim_province_reporting_area_source AS source
ON target.reporting_area_key = source.reporting_area_key
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

MERGE INTO `03-gold`.dim_province_reporting_area_boundary AS target
USING (
    SELECT
        boundary_key, reporting_area_key,
        ST_SETSRID(boundary_geometry, 4326) AS boundary_geometry,
        boundary_version, coordinate_reference_system,
        geometry_valid_flag, boundary_status, run_id,
        source_load_ts, psgc_run_id, boundary_run_id
    FROM dim_province_reporting_area_boundary_source
) AS source
ON target.boundary_key = source.boundary_key
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

SELECT
    area.reporting_area_type, area.reporting_area_name, area.region_key,
    boundary.boundary_status, area.run_id
FROM `03-gold`.dim_province_reporting_area AS area
INNER JOIN `03-gold`.dim_province_reporting_area_boundary AS boundary
    ON area.reporting_area_key = boundary.reporting_area_key
WHERE area.run_id = (SELECT dimension_run_id FROM gold_reporting_area_versions)
ORDER BY area.reporting_area_type, area.reporting_area_name;


# Summary

Gold now has a semantically accurate reporting-area dimension and boundary table. Region remains a required parent key. Unsafe or unavailable geometry remains NULL with an explicit status.
